# Real Data Experiments: Waterbirds & SpurBreast

**Datasets:**
- **Waterbirds** — bird type (waterbird/landbird) is the label, background (water/land) is the spurious shortcut
- **SpurBreast** — breast MRI classification with two spurious correlations: magnetic field strength (global feature) and image orientation (local feature)

**Experiment structure for each dataset:**
1. Hyperparameter tuning via linear probing
2. Hyperparameter tuning via finetuning
3. Testing via linear probing (using best HPs)
4. Testing via finetuning (using best HPs)

---
## 1. Setup & Imports

In [ ]:
# Install dependencies
import subprocess, sys

for pkg in ['seaborn', 'pandas']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/shortcut_experiments'
    IN_COLAB = True
except ImportError:
    DRIVE_ROOT = './shortcut_experiments'
    IN_COLAB = False

import os
os.makedirs(f'{DRIVE_ROOT}/csv', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/logs', exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')
print(f'Running in Colab: {IN_COLAB}')

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import time, json, os, warnings, glob, copy
from collections import OrderedDict, Counter
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
from torchvision import transforms, models
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

---
## 2. Loss Functions

All loss functions adapted for real data (labels in {0, 1}).

In [ ]:
class LogLoss(nn.Module):
    def forward(self, logits, labels):
        # labels in {0, 1}
        return nn.functional.binary_cross_entropy_with_logits(logits, labels.float())


class SpectralDecoupling(nn.Module):
    """SD loss: log-loss + λ * |f(x)|^2"""
    def __init__(self, lam=0.1):
        super().__init__()
        self.lam = lam

    def forward(self, logits, labels):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, labels.float())
        penalty = self.lam * (logits ** 2).mean()
        return bce + penalty


class MargLogLoss(nn.Module):
    """Marg-Log loss with per-class targets: log-loss + λ * log(1 + |f - γ_y|^2)

    Per-class targets γ_0, γ_1 handle label imbalance.
    """
    def __init__(self, lam=0.1, gamma_0=0.0, gamma_1=1.0):
        super().__init__()
        self.lam = lam
        self.gamma_0 = gamma_0
        self.gamma_1 = gamma_1

    def forward(self, logits, labels):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, labels.float())
        targets = torch.where(labels == 1,
                              torch.tensor(self.gamma_1, device=logits.device),
                              torch.tensor(self.gamma_0, device=logits.device))
        penalty = self.lam * torch.log(1 + (logits.squeeze() - targets) ** 2).mean()
        return bce + penalty


class SigmaDampLoss(nn.Module):
    """σ-Damp loss with per-class temperatures.

    ℓ_σ-damp(y, f) = ℓ_log(T_y * 1.278 * yf * (1 - σ(1.278 * yf)))
    """
    def __init__(self, T_0=1.0, T_1=2.0):
        super().__init__()
        self.T_0 = T_0
        self.T_1 = T_1

    def forward(self, logits, labels):
        signs = 2 * labels.float() - 1  # {-1, +1}
        margins = signs * logits.squeeze()

        T = torch.where(labels == 1,
                        torch.tensor(self.T_1, device=logits.device),
                        torch.tensor(self.T_0, device=logits.device))

        scaled_m = 1.278 * margins
        damped_input = T * scaled_m * (1 - torch.sigmoid(scaled_m))

        loss = torch.log(1 + torch.exp(-damped_input)).mean()
        return loss

---
## 3. Shared Helper Functions

Evaluation, subsampling, and utility functions used across both datasets.

In [ ]:
# ── Evaluation helpers ──

def compute_group_metrics(preds, labels, groups, group_names=None):
    """Compute per-group accuracy, sizes, and worst-group accuracy."""
    unique_groups = np.unique(groups)
    results = {}
    for g in unique_groups:
        mask = groups == g
        acc = np.mean(preds[mask] == labels[mask])
        name = group_names[g] if group_names else f'group_{g}'
        results[name] = {
            'accuracy': acc,
            'size': int(np.sum(mask)),
            'fraction': np.mean(mask),
        }
    wga = min(r['accuracy'] for r in results.values())
    avg = np.mean(preds == labels)
    return {'per_group': results, 'wga': wga, 'avg_acc': avg}


def print_group_summary(metrics, dataset_name=''):
    """Pretty-print group metrics."""
    print(f"\n{'='*60}")
    print(f"  {dataset_name} Results")
    print(f"  Average Acc: {metrics['avg_acc']:.4f}  |  WGA: {metrics['wga']:.4f}")
    print(f"{'='*60}")
    for name, info in metrics['per_group'].items():
        print(f"  {name:30s}  n={info['size']:5d} ({info['fraction']:.3f})  acc={info['accuracy']:.4f}")


# ── Subsampling strategies for real data ──

def real_data_gb_downsample(X, y, groups, seed=0):
    """Group-balanced downsampling: match smallest group size."""
    rng = np.random.RandomState(seed)
    unique_groups = np.unique(groups)
    min_size = min(np.sum(groups == g) for g in unique_groups)
    indices = []
    for g in unique_groups:
        g_idx = np.where(groups == g)[0]
        chosen = rng.choice(g_idx, size=min_size, replace=False)
        indices.extend(chosen)
    indices = np.array(indices)
    rng.shuffle(indices)
    return X[indices], y[indices], groups[indices], indices


def real_data_gb_oversample(X, y, groups, seed=0):
    """Group-balanced oversampling: match largest group size."""
    rng = np.random.RandomState(seed)
    unique_groups = np.unique(groups)
    max_size = max(np.sum(groups == g) for g in unique_groups)
    indices = []
    for g in unique_groups:
        g_idx = np.where(groups == g)[0]
        chosen = rng.choice(g_idx, size=max_size, replace=True)
        indices.extend(chosen)
    indices = np.array(indices)
    rng.shuffle(indices)
    return X[indices], y[indices], groups[indices], indices


def real_data_random_subsample(X, y, groups, ratio=0.5, seed=0):
    """Random subsampling preserving group ratios."""
    rng = np.random.RandomState(seed)
    n_keep = int(len(y) * ratio)
    indices = rng.choice(len(y), size=n_keep, replace=False)
    return X[indices], y[indices], groups[indices], indices


# ── Save/load utilities ──

def save_csv(df, name):
    """Save DataFrame to Drive CSV directory."""
    path = f'{DRIVE_ROOT}/csv/{name}'
    df.to_csv(path, index=False)
    print(f'  Saved {path} ({len(df)} rows)')


def save_checkpoint(model, metadata, name):
    """Save model weights + metadata dict to Drive."""
    path = f'{DRIVE_ROOT}/checkpoints/{name}.pt'
    torch.save({
        'model_state_dict': model.state_dict(),
        'metadata': metadata,
    }, path)

---
# Waterbirds Experiments

## 4. Waterbirds — Data Loading

In [ ]:
# Install WILDS for Waterbirds
!pip install wilds

In [ ]:
# Extract dataset to correct location (run once)
!tar -xf /content/drive/MyDrive/waterbirds_v1.0/archive.tar.gz -C /content/drive/MyDrive/waterbirds_v1.0/
!ls -l /content/drive/MyDrive/waterbirds_v1.0/metadata.csv

In [ ]:
def load_waterbirds(root_dir='/content/drive/MyDrive'):
    """Load Waterbirds dataset from WILDS. Returns embeddings + labels + groups."""
    from wilds import get_dataset
    dataset = get_dataset(dataset='waterbirds', root_dir=root_dir, download=False)
    resnet = models.resnet50(pretrained=True)
    resnet.fc = nn.Identity()  # remove classification head, keep embeddings
    resnet.eval().to(device)
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])
    splits = {}
    for split_name in ['train', 'test']:
        subset = dataset.get_subset(split_name, transform=transform)
        loader = torch.utils.data.DataLoader(subset, batch_size=64, num_workers=2)
        all_features, all_labels, all_groups = [], [], []
        with torch.no_grad():
            for batch in loader:
                x, y_true, metadata = batch
                features = resnet(x.to(device)).cpu().numpy()
                all_features.append(features)
                all_labels.append(y_true.numpy())
                place = metadata[:, 0].numpy()
                groups = 2 * y_true.numpy() + place  # 4 groups
                all_groups.append(groups)
        splits[split_name] = {
            'X': np.concatenate(all_features).astype(np.float32),
            'y': np.concatenate(all_labels).astype(np.float32),
            'groups': np.concatenate(all_groups).astype(int),
        }
    group_names = {0: 'landbird_land', 1: 'landbird_water', 2: 'waterbird_land', 3: 'waterbird_water'}
    return splits, group_names


def load_waterbirds_wilds(data_dir='./data'):
    """Load Waterbirds via WILDS with standard ImageNet transforms (for finetuning)."""
    from wilds import get_dataset
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    dataset = get_dataset(dataset='waterbirds', root_dir=data_dir, download=False)
    splits = {}
    for split_name in ['train', 'val', 'test']:
        split_data = dataset.get_subset(split_name, transform=transform)
        splits[split_name] = split_data
    return dataset, splits

---
## 5. Waterbirds — Linear Probe Infrastructure

In [ ]:
def extract_embeddings(data_dir='./data', device='cuda'):
    """Extract ResNet-50 embeddings for Waterbirds (train/val/test)."""
    dataset, splits_raw = load_waterbirds_wilds(data_dir)

    backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    backbone.fc = nn.Identity()
    backbone = backbone.to(device)
    backbone.eval()

    embeddings = {}
    for split_name in ['train', 'val', 'test']:
        split_data = splits_raw[split_name]
        loader = DataLoader(split_data, batch_size=64, shuffle=False, num_workers=2)

        all_X, all_y, all_groups = [], [], []
        with torch.no_grad():
            for x, y, metadata in loader:
                x = x.to(device)
                feats = backbone(x).cpu().numpy()
                all_X.append(feats)
                all_y.append(y.numpy())
                groups = 2 * y + metadata[:, 0]
                all_groups.append(groups.numpy())

        embeddings[split_name] = {
            'X': np.concatenate(all_X),
            'y': np.concatenate(all_y),
            'groups': np.concatenate(all_groups),
        }

    return embeddings


def run_linear_probe_fixed(X_train, y_train, groups_train,
                           X_val, y_val, groups_val,
                           X_test, y_test, groups_test,
                           loss_fn, lr=0.01, epochs=5000,
                           device='cuda', use_bias=True):
    """Train linear probe with validation-based early stopping."""
    d = X_train.shape[1]

    X_tr = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_tr = torch.tensor(y_train, dtype=torch.long, device=device)
    g_tr = torch.tensor(groups_train, dtype=torch.long, device=device)

    X_v = torch.tensor(X_val, dtype=torch.float32, device=device)
    y_v = torch.tensor(y_val, dtype=torch.long, device=device)
    g_v = torch.tensor(groups_val, dtype=torch.long, device=device)

    X_te = torch.tensor(X_test, dtype=torch.float32, device=device)
    y_te = torch.tensor(y_test, dtype=torch.long, device=device)
    g_te = torch.tensor(groups_test, dtype=torch.long, device=device)

    model = nn.Linear(d, 1, bias=use_bias).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val_wga = -1
    best_state = None
    patience = 500
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(X_tr).squeeze(-1)
        loss = loss_fn(logits, y_tr)
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 50 == 0:
            model.eval()
            with torch.no_grad():
                val_logits = model(X_v).squeeze(-1)
                val_preds = (val_logits > 0).long()
                val_correct = (val_preds == y_v).float()

                val_group_accs = []
                for g in g_v.unique():
                    mask = g_v == g
                    if mask.sum() > 0:
                        val_group_accs.append(val_correct[mask].mean().item())

                val_wga = min(val_group_accs) if val_group_accs else 0.0

                if val_wga > best_val_wga:
                    best_val_wga = val_wga
                    best_state = copy.deepcopy(model.state_dict())
                    patience_counter = 0
                else:
                    patience_counter += 50

                if patience_counter >= patience:
                    break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        test_logits = model(X_te).squeeze(-1)
        test_preds = (test_logits > 0).long()
        test_correct = (test_preds == y_te).float()

        test_avg = test_correct.mean().item()
        test_group_accs = {}
        for g in g_te.unique():
            mask = g_te == g
            test_group_accs[g.item()] = test_correct[mask].mean().item()
        test_wga = min(test_group_accs.values())

        train_logits = model(X_tr).squeeze(-1)
        train_loss = loss_fn(train_logits, y_tr).item()
        train_preds = (train_logits > 0).long()
        train_acc = (train_preds == y_tr).float().mean().item()

    return {
        'test_wga': test_wga,
        'test_avg': test_avg,
        'test_groups': test_group_accs,
        'val_wga': best_val_wga,
        'train_loss': train_loss,
        'train_acc': train_acc,
    }


# ── Subsampling helpers for embeddings ──

def gb_downsample(X, y, groups):
    """Group-balanced downsampling: downsample each group to min group size."""
    unique_groups = np.unique(groups)
    min_size = min(np.sum(groups == g) for g in unique_groups)
    indices = []
    for g in unique_groups:
        g_idx = np.where(groups == g)[0]
        chosen = np.random.choice(g_idx, size=min_size, replace=False)
        indices.extend(chosen)
    indices = np.array(indices)
    return X[indices], y[indices], groups[indices]


def gb_oversample(X, y, groups):
    """Group-balanced oversampling: oversample each group to max group size."""
    unique_groups = np.unique(groups)
    max_size = max(np.sum(groups == g) for g in unique_groups)
    indices = []
    for g in unique_groups:
        g_idx = np.where(groups == g)[0]
        if len(g_idx) < max_size:
            chosen = np.random.choice(g_idx, size=max_size, replace=True)
        else:
            chosen = g_idx
        indices.extend(chosen)
    indices = np.array(indices)
    return X[indices], y[indices], groups[indices]


def get_balanced_subset(X, y, groups, k_per_group):
    """Generalized group-balanced sampler. Downsamples or oversamples each group to k_per_group."""
    unique_groups = np.unique(groups)
    indices = []
    for g in unique_groups:
        g_idx = np.where(groups == g)[0]
        if len(g_idx) >= k_per_group:
            chosen = np.random.choice(g_idx, size=k_per_group, replace=False)
        else:
            chosen = np.random.choice(g_idx, size=k_per_group, replace=True)
        indices.extend(chosen)
    indices = np.array(indices)
    shuffle_idx = np.random.permutation(len(indices))
    return X[indices[shuffle_idx]], y[indices[shuffle_idx]], groups[indices[shuffle_idx]]

---
## 6. Waterbirds — Hyperparameter Tuning (Linear Probe)

In [ ]:
# Stable SD variant for Waterbirds: uses softplus for numerical stability
class SpectralDecoupling(nn.Module):
    """Spectral Decoupling: log-loss + lambda * ||f||^2."""
    def __init__(self, lam=0.1):
        super().__init__()
        self.lam = lam
    def forward(self, logits, labels):
        f = logits.squeeze()
        targets = 2 * labels.float() - 1  # convert {0,1} -> {-1,1}
        loss = torch.nn.functional.softplus(-targets * f).mean()
        return loss + self.lam * (f ** 2).mean()

WATERBIRDS = True
if WATERBIRDS:
    splits, group_names = load_waterbirds()
    X_tr = splits['train']['X']
    y_tr = splits['train']['y']
    X_te = splits['test']['X']
    y_te = splits['test']['y']
    g_te = splits['test']['groups']

    configs = [
        ('SD_lam005',  SpectralDecoupling(lam=0.05)),
        ('SD_lam01',   SpectralDecoupling(lam=0.1)),
        ('SD_lam05',   SpectralDecoupling(lam=0.5)),
        ('ML_lam001_g2',  MargLogLoss(lam=0.01, gamma_0=0, gamma_1=2)),
        ('ML_lam005_g2',  MargLogLoss(lam=0.05, gamma_0=0, gamma_1=2)),
        ('ML_lam01_g2',   MargLogLoss(lam=0.1,  gamma_0=0, gamma_1=2)),
        ('ML_lam05_g2',   MargLogLoss(lam=0.5,  gamma_0=0, gamma_1=2)),
        ('ML_lam1_g2',    MargLogLoss(lam=1.0,  gamma_0=0, gamma_1=2)),
        ('ML_g1_05',   MargLogLoss(lam=0.1, gamma_0=0, gamma_1=0.5)),
        ('ML_g1_1',    MargLogLoss(lam=0.1, gamma_0=0, gamma_1=1)),
        ('ML_g1_2',    MargLogLoss(lam=0.1, gamma_0=0, gamma_1=2)),
        ('σD_T1_05',  SigmaDampLoss(T_0=1, T_1=0.5)),
        ('σD_T1_1',   SigmaDampLoss(T_0=1, T_1=1)),
        ('σD_T1_2',   SigmaDampLoss(T_0=1, T_1=2)),
        ('σD_T1_3',   SigmaDampLoss(T_0=1, T_1=3)),
        ('σD_T0_05',  SigmaDampLoss(T_0=0.5, T_1=2)),
        ('σD_T0_1',   SigmaDampLoss(T_0=1,   T_1=2)),
        ('σD_T0_2',   SigmaDampLoss(T_0=2,   T_1=2)),
    ]
    d = X_tr.shape[1]
    hp_results = []

    for config_name, loss_factory in configs:
        print(f"\n{'='*60}")
        print(f"  Waterbirds HP: {config_name}")
        print(f"{'='*60}")

        torch.manual_seed(0)
        model = nn.Linear(d, 1, bias=False).to(device)
        optimizer = optim.Adam(model.parameters(), lr=0.01)

        Xt = torch.tensor(X_tr, dtype=torch.float32).to(device)
        yt = torch.tensor(y_tr, dtype=torch.float32).to(device)

        loss_fn = loss_factory
        for epoch in range(5000):
            optimizer.zero_grad()
            logits = model(Xt).squeeze()
            loss = loss_fn(logits, yt)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            test_logits = model(torch.tensor(X_te, dtype=torch.float32).to(device)).squeeze().cpu().numpy()
            test_preds = (test_logits > 0).astype(float)

        metrics = compute_group_metrics(test_preds, y_te, g_te, group_names)
        print(f"  => avg_acc={metrics['avg_acc']:.3f} | wga={metrics['wga']:.3f}")

        hp_results.append({
            'config': config_name,
            'avg_acc': metrics['avg_acc'],
            'wga': metrics['wga'],
        })

    df_hp_wb = pd.DataFrame(hp_results)
    save_csv(df_hp_wb, 'results_waterbirds_hyperparam.csv')
    print(df_hp_wb.to_string(index=False, float_format='{:.3f}'.format))

---
## 7. Waterbirds — Hyperparameter Tuning (Finetuning)

In [ ]:
# ── Finetuning helpers ──

def train_finetune_one_epoch(model, loader, optimizer, loss_fn, device):
    """Train one epoch of full finetuning."""
    model.train()
    total_loss = 0
    n_samples = 0
    for x, y, metadata in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x).squeeze(-1)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        n_samples += x.size(0)
    return total_loss / n_samples


def evaluate_finetune(model, loader, device):
    """Evaluate with per-group accuracy (groups = 2*y + background)."""
    model.eval()
    all_preds, all_labels, all_groups = [], [], []
    with torch.no_grad():
        for x, y, metadata in loader:
            x = x.to(device)
            logits = model(x).squeeze(-1)
            preds = (logits > 0).long().cpu()
            all_preds.append(preds)
            all_labels.append(y)
            groups = 2 * y + metadata[:, 0]  # 4 groups
            all_groups.append(groups)

    preds = torch.cat(all_preds)
    labels = torch.cat(all_labels)
    groups = torch.cat(all_groups)

    correct = (preds == labels).float()
    group_accs = {}
    for g in groups.unique():
        mask = groups == g
        group_accs[g.item()] = correct[mask].mean().item()

    avg_acc = correct.mean().item()
    wga = min(group_accs.values()) if group_accs else 0.0
    return avg_acc, wga, group_accs

In [ ]:
# ── Waterbirds HP Tuning (Finetuning) ──

from wilds import get_dataset
import torchvision.models as models

DATA_DIR = '/content/drive/MyDrive'
dataset, splits = load_waterbirds_wilds(DATA_DIR)

train_loader = DataLoader(splits['train'], batch_size=64, shuffle=True, num_workers=4)
val_loader = DataLoader(splits['val'], batch_size=64, shuffle=False, num_workers=4)
test_loader = DataLoader(splits['test'], batch_size=64, shuffle=False, num_workers=4)

configs = [
    ('SD_lam005',  SpectralDecoupling(lam=0.05)),
    ('SD_lam01',   SpectralDecoupling(lam=0.1)),
    ('SD_lam05',   SpectralDecoupling(lam=0.5)),
    ('ML_lam01',   MargLogLoss(lam=0.1, gamma_0=0, gamma_1=2)),
    ('ML_lam05',   MargLogLoss(lam=0.5, gamma_0=0, gamma_1=2)),
    ('ML_lam1',    MargLogLoss(lam=1.0, gamma_0=0, gamma_1=2)),
    ('ML_g1_1',    MargLogLoss(lam=0.1, gamma_0=0, gamma_1=1)),
    ('ML_g1_4',    MargLogLoss(lam=0.1, gamma_0=0, gamma_1=4)),
    ('σD_T05',     SigmaDampLoss(T_0=0.5, T_1=2)),
    ('σD_T1',      SigmaDampLoss(T_0=1, T_1=2)),
    ('σD_T1_3',    SigmaDampLoss(T_0=1, T_1=3)),
    ('σD_T1_05',   SigmaDampLoss(T_0=1, T_1=0.5)),
]

hp_results = []

for config_name, loss_fn in configs:
    print(f"\n{'='*60}")
    print(f"  WB Finetune HP: {config_name}")
    print(f"{'='*60}")

    torch.manual_seed(0)
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    model.fc = nn.Linear(2048, 1)
    model = model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=1e-5, weight_decay=0.1)

    best_val_wga = -1
    best_state = None
    patience_counter = 0

    for epoch in range(50):
        train_finetune_one_epoch(model, train_loader, optimizer, loss_fn, device)
        _, val_wga, _ = evaluate_finetune(model, val_loader, device)

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/50 | val_wga={val_wga:.3f}")

        if val_wga > best_val_wga:
            best_val_wga = val_wga
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 15:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    if best_state:
        model.load_state_dict(best_state)

    test_avg, test_wga, test_groups = evaluate_finetune(model, test_loader, device)
    print(f"  => val_wga={best_val_wga:.3f} | test_wga={test_wga:.3f} | test_avg={test_avg:.3f}")

    hp_results.append({
        'config': config_name,
        'val_wga': best_val_wga,
        'test_wga': test_wga,
        'test_avg': test_avg,
    })

df_hp_wb_ft = pd.DataFrame(hp_results)
df_hp_wb_ft.to_csv(f'{DRIVE_ROOT}/csv/results_waterbirds_hp_finetune.csv', index=False)
print(df_hp_wb_ft.to_string(index=False, float_format='{:.3f}'.format))

---
## 8. Waterbirds — Testing (Linear Probe)

Using best hyperparameters from Section 6.

In [ ]:
def run_fixed_linear_probe_experiment(embeddings, device='cuda'):
    """Run the fixed linear probe experiment with single robust hyperparameters."""
    X_tr = embeddings['train']['X']
    y_tr = embeddings['train']['y']
    g_tr = embeddings['train']['groups']
    X_val = embeddings['val']['X']
    y_val = embeddings['val']['y']
    g_val = embeddings['val']['groups']
    X_te = embeddings['test']['X']
    y_te = embeddings['test']['y']
    g_te = embeddings['test']['groups']

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)
    X_te = scaler.transform(X_te).astype(np.float32)

    print(f"Data: train={len(y_tr)}, val={len(y_val)}, test={len(y_te)}, d={X_tr.shape[1]}")
    print(f"d/n_train = {X_tr.shape[1] / len(y_tr):.3f}")

    lr_grid = [0.01]

    method_configs = {
        'ERM': [{'loss_fn': LogLoss(), 'name': 'ERM'}],
        'SD': [{'loss_fn': SpectralDecoupling(lam=0.1), 'name': 'SD(λ=0.1)'}],
        'Marg-Log': [{'loss_fn': MargLogLoss(lam=0.1, gamma_0=0, gamma_1=2), 'name': 'ML(γ0=0,γ1=2)'}],
        'σ-damp': [{'loss_fn': SigmaDampLoss(T_0=1, T_1=2), 'name': 'σD(T0=1,T1=2)'}],
    }

    all_results = []

    for method_name, configs in method_configs.items():
        best_val_wga = -1
        best_result = None

        for config in configs:
            for lr in lr_grid:
                for seed in range(3):
                    torch.manual_seed(seed)
                    np.random.seed(seed)

                    result = run_linear_probe_fixed(
                        X_tr, y_tr, g_tr,
                        X_val, y_val, g_val,
                        X_te, y_te, g_te,
                        loss_fn=config['loss_fn'],
                        lr=lr, epochs=5000,
                        device=device, use_bias=True
                    )
                    result['method'] = method_name
                    result['config'] = config['name']
                    result['lr'] = lr
                    result['seed'] = seed
                    all_results.append(result)

                    if result['val_wga'] > best_val_wga:
                        best_val_wga = result['val_wga']
                        best_result = result

        print(f"\n{'='*60}")
        print(f"  BEST {method_name}: val_wga={best_result['val_wga']:.3f} "
              f"test_wga={best_result['test_wga']:.3f}")
        print(f"  Config: {best_result['config']}, lr={best_result['lr']}")
        print(f"  Per-group: {best_result['test_groups']}")
        print(f"{'='*60}")

    return pd.DataFrame(all_results)

In [ ]:
# Run the linear probe test
MODE = 'fix_linear'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if MODE == 'fix_linear':
    print("Extracting embeddings...")
    embeddings = extract_embeddings(DATA_DIR, DEVICE)
    print("Running Fixed Linear Probe Experiment...")
    df = run_fixed_linear_probe_experiment(embeddings, DEVICE)
    df.to_csv('results_waterbirds_fixed_linear.csv', index=False)

elif MODE == 'fix_subsample':
    embeddings = extract_embeddings(DATA_DIR, DEVICE)
    df = run_fixed_with_subsampling(embeddings, DEVICE)
    df.to_csv('results_waterbirds_fixed_subsample.csv', index=False)

---
## 9. Waterbirds — Testing (Finetuning)

Using best hyperparameters from Section 7.

In [ ]:
# Best HPs from finetuning sweep
methods = {
    'ERM':      LogLoss(),
    'SD':       SpectralDecoupling(lam=0.05),                      # Best SD WGA (0.668)
    'Marg-Log': MargLogLoss(lam=4.0, gamma_0=0.0, gamma_1=1.0),    # Best ML WGA (0.734)
    'σ-damp':   SigmaDampLoss(T_0=1.0, T_1=3.0),                   # Best σD WGA (0.621)
}

DATA_DIR = '/content/drive/MyDrive'
dataset, splits = load_waterbirds_wilds(DATA_DIR)

train_loader = DataLoader(splits['train'], batch_size=64, shuffle=True, num_workers=4)
val_loader = DataLoader(splits['val'], batch_size=64, shuffle=False, num_workers=4)
test_loader = DataLoader(splits['test'], batch_size=64, shuffle=False, num_workers=4)

# Get train metadata for subsampling
train_data = splits['train']
train_labels = np.array([train_data[i][1] if not torch.is_tensor(train_data[i][1]) else train_data[i][1].item() for i in range(len(train_data))])
train_groups = np.array([2 * train_labels[i] + (train_data[i][2][0].item() if torch.is_tensor(train_data[i][2][0]) else train_data[i][2][0]) for i in range(len(train_data))])

val_loader = DataLoader(splits['val'], batch_size=64, shuffle=False, num_workers=4)
test_loader = DataLoader(splits['test'], batch_size=64, shuffle=False, num_workers=4)

conditions = {
    'random_005': ('random', 0.05),
    'random_010': ('random', 0.10),
    'random_025': ('random', 0.25),
    'random_050': ('random', 0.50),
    'full':       ('random', 1.0),
    'gb_005':     ('balanced', 0.05),
    'gb_010':     ('balanced', 0.10),
    'gb_025':     ('balanced', 0.25),
    'gb_050':     ('balanced', 0.50),
    'gb_full':    ('balanced', 1.0),
}

all_results = []

for cond_name, (sampling, ratio) in conditions.items():
    rng = np.random.RandomState(42)
    n = len(train_labels)

    if sampling == 'random':
        n_keep = max(int(n * ratio), 2)
        indices = rng.choice(n, size=n_keep, replace=False)
    else:
        unique_g = np.unique(train_groups)
        min_size = min(np.sum(train_groups == g) for g in unique_g)
        target_per_group = max(int(min_size * ratio), 2)
        indices = []
        for g in unique_g:
            g_idx = np.where(train_groups == g)[0]
            chosen = rng.choice(g_idx, size=min(target_per_group, len(g_idx)), replace=False)
            indices.extend(chosen)
        indices = np.array(indices)
        rng.shuffle(indices)

    train_subset = Subset(train_data, indices)
    n_train = len(indices)
    d_over_n = 2048 / n_train
    shortcut_ratio = np.mean(np.isin(train_groups[indices], [0, 3]))

    train_loader_sub = DataLoader(train_subset, batch_size=64, shuffle=True, num_workers=4)

    for method_name, loss_fn in methods.items():
        print(f"\n{'='*60}")
        print(f"  {cond_name} | {method_name} | n={n_train} | d/n={d_over_n:.3f} | ρ={shortcut_ratio:.3f}")
        print(f"{'='*60}")

        torch.manual_seed(0)
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(2048, 1)
        model = model.to(device)

        optimizer = optim.Adam(model.parameters(), lr=1e-5, weight_decay=0.1)

        best_val_wga = -1
        best_state = None
        patience_counter = 0

        for epoch in range(50):
            model.train()
            for x, y, metadata in train_loader_sub:
                x, y = x.to(device), y.to(device).float()
                optimizer.zero_grad()
                logits = model(x).squeeze(-1)
                loss = loss_fn(logits, y)
                loss.backward()
                optimizer.step()

            _, val_wga, _ = evaluate_finetune(model, val_loader, device)

            if (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1}/50 | val_wga={val_wga:.3f}")

            if val_wga > best_val_wga:
                best_val_wga = val_wga
                best_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= 15:
                    print(f"  Early stopping at epoch {epoch+1}")
                    break

        if best_state:
            model.load_state_dict(best_state)

        test_avg, test_wga, _ = evaluate_finetune(model, test_loader, device)
        print(f"  => test_wga={test_wga:.3f} | test_avg={test_avg:.3f}")

        all_results.append({
            'condition': cond_name,
            'sampling': sampling,
            'method': method_name,
            'n_train': n_train,
            'd_over_n': d_over_n,
            'shortcut_ratio': shortcut_ratio,
            'val_wga': best_val_wga,
            'test_wga': test_wga,
            'test_avg': test_avg,
        })

DRIVE_ROOT = '/content/drive/MyDrive'
df_wb_ft = pd.DataFrame(all_results)
df_wb_ft.to_csv(f'{DRIVE_ROOT}/csv/results_waterbirds_finetuning.csv', index=False)
print(df_wb_ft.to_string(index=False, float_format='{:.3f}'.format))

---
# SpurBreast Experiments

## 10. SpurBreast — Data Loading & Feature Extraction

SpurBreast has two shortcuts:
- **z1**: Field Strength (1.5T vs 3T) — real, from DUKE metadata
- **z2**: Vertical Flip — injected with correlation ρ₂

Requires downloaded SpurBreast from https://github.com/utkuozbulak/spurbreast (field_strength folder).

In [ ]:
# ── DUKE field strength metadata ──

def get_file_paths(folder, label):
    paths = []
    for ext in ['*.png', '*.jpg', '*.jpeg']:
        paths.extend(glob.glob(os.path.join(folder, ext)))
    return [(p, label) for p in sorted(paths)]


def load_duke_field_strength(clinical_xlsx_path):
    """Load DUKE clinical excel, return patient_num -> field_strength_binary mapping."""
    if not os.path.exists(clinical_xlsx_path):
        print(f"WARNING: Duke clinical file not found at {clinical_xlsx_path}. Assuming all 1.5T.")
        return {}
    duke_df = pd.read_excel(clinical_xlsx_path, header=1, skiprows=[2])
    fs_map = {}
    for _, row in duke_df.iterrows():
        pat_id = str(row['Patient ID'])
        pat_num = int(pat_id.split('_')[-1])
        fs_code = int(row['Field Strength (Tesla)'])
        fs_map[pat_num] = 1 if fs_code >= 2 else 0  # 0=1.5T, 1=3T
    print(f"DUKE metadata: {len(fs_map)} patients, 1.5T={sum(1 for v in fs_map.values() if v==0)}, 3T={sum(1 for v in fs_map.values() if v==1)}")
    return fs_map


def bias_field_strength(image_paths, y_arr, z1_arr, z2_arr, rho1=0.9, seed=42):
    """Subsample training data so z1 correlates with y at rate rho1."""
    rng = np.random.RandomState(seed)

    agree_idx = np.where(y_arr == z1_arr)[0]
    disagree_idx = np.where(y_arr != z1_arr)[0]

    n_agree_from_disagree = int(len(agree_idx) * (1 - rho1) / rho1)
    n_disagree_from_agree = int(len(disagree_idx) * rho1 / (1 - rho1))

    if n_agree_from_disagree <= len(disagree_idx):
        keep_agree = agree_idx
        keep_disagree = rng.choice(disagree_idx, size=n_agree_from_disagree, replace=False)
    else:
        keep_agree = rng.choice(agree_idx, size=n_disagree_from_agree, replace=False)
        keep_disagree = disagree_idx

    keep_idx = np.concatenate([keep_agree, keep_disagree])
    rng.shuffle(keep_idx)

    new_paths = [image_paths[i] for i in keep_idx]
    new_y = y_arr[keep_idx]
    new_z1 = z1_arr[keep_idx]
    new_z2 = z2_arr[keep_idx]

    actual_corr = np.mean(new_y == new_z1)
    print(f"  Field strength biasing: {len(y_arr)} -> {len(new_y)} images")
    print(f"  z1 correlation: {np.mean(y_arr == z1_arr):.3f} -> {actual_corr:.3f}")

    return new_paths, new_y, new_z1, new_z2

In [ ]:
# ── Feature extraction for linear probe experiments ──

def extract_spurbreast_features(fs_dir, baseline_dir, clinical_xlsx_path, rho2=0.7, flip_seed=42, device='cuda'):
    """Extract ResNet-50 embeddings with two-shortcut structure."""
    fs_map = load_duke_field_strength(clinical_xlsx_path)

    def get_fs_label(filename):
        try:
            pat_num = int(os.path.basename(filename).split('-')[1])
            return fs_map.get(pat_num, 0)
        except:
            return 0

    resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    resnet.fc = nn.Identity()
    resnet = resnet.to(device).eval()

    def extract_from_folder(split_dir):
        pos_paths = get_file_paths(os.path.join(split_dir, '1'), 1)
        neg_paths = get_file_paths(os.path.join(split_dir, '0'), 0)
        all_paths = pos_paths + neg_paths

        if len(all_paths) == 0:
            return None, None

        all_features, batch, labels = [], [], []
        for i, (path, label) in enumerate(all_paths):
            im = Image.open(path).convert('RGB').resize((224, 224))
            im_tensor = torch.tensor(np.array(im), dtype=torch.float32).permute(2, 0, 1) / 255.0
            batch.append(im_tensor)
            labels.append(label)

            if len(batch) == 64 or i == len(all_paths) - 1:
                with torch.no_grad():
                    feats = resnet(torch.stack(batch).to(device)).cpu().numpy()
                all_features.append(feats)
                batch = []

        return np.concatenate(all_features).astype(np.float32), np.array(labels, dtype=np.float32)

    def extract_with_flip(split_dir, rho2, seed):
        rng = np.random.RandomState(seed)
        pos_paths = get_file_paths(os.path.join(split_dir, '1'), 1)
        neg_paths = get_file_paths(os.path.join(split_dir, '0'), 0)
        all_paths = pos_paths + neg_paths

        if len(all_paths) == 0:
            raise ValueError(f"No images found in {split_dir}")

        all_features, batch, y_list, z1_list, z2_list = [], [], [], [], []

        for i, (path, label) in enumerate(all_paths):
            fname = os.path.basename(path)
            im = Image.open(path).convert('RGB').resize((224, 224))

            z1_val = get_fs_label(fname)
            z2_val = label if rng.random() < rho2 else 1 - label

            if z2_val == 1:
                im = im.transpose(Image.FLIP_TOP_BOTTOM)

            im_tensor = torch.tensor(np.array(im), dtype=torch.float32).permute(2, 0, 1) / 255.0
            batch.append(im_tensor)
            y_list.append(label)
            z1_list.append(z1_val)
            z2_list.append(z2_val)

            if len(batch) == 64 or i == len(all_paths) - 1:
                with torch.no_grad():
                    feats = resnet(torch.stack(batch).to(device)).cpu().numpy()
                all_features.append(feats)
                batch = []

        return np.concatenate(all_features).astype(np.float32), np.array(y_list, dtype=np.float32), np.array(z1_list), np.array(z2_list)

    print("Extracting Training Set (with Flips & Field Strength)...")
    X_tr, y_tr, z1_tr, z2_tr = extract_with_flip(os.path.join(fs_dir, 'training'), rho2, flip_seed)

    agree1 = (y_tr == z1_tr).astype(int)
    agree2 = (y_tr == z2_tr).astype(int)
    groups_tr = 2 * agree1 + agree2

    print("Extracting Test Set (Baseline Unbiased)...")
    X_te, y_te = extract_from_folder(os.path.join(baseline_dir, 'test'))
    groups_te = y_te.astype(int)

    # 80/20 train/val split
    np.random.seed(42)
    indices = np.random.permutation(len(y_tr))
    split = int(len(y_tr) * 0.8)
    tr_idx, val_idx = indices[:split], indices[split:]

    embeddings = {
        'train': {'X': X_tr[tr_idx], 'y': y_tr[tr_idx], 'groups': groups_tr[tr_idx]},
        'val': {'X': X_tr[val_idx], 'y': y_tr[val_idx], 'groups': groups_tr[val_idx]},
        'test': {'X': X_te, 'y': y_te, 'groups': groups_te}
    }
    return embeddings

---
## 11. SpurBreast — Hyperparameter Tuning (Linear Probe)

In [ ]:
# Redefine SD back to standard BCE version for SpurBreast
# (Waterbirds used a softplus variant above for numerical stability)
class SpectralDecoupling(nn.Module):
    """SD loss: log-loss + lambda * |f(x)|^2"""
    def __init__(self, lam=0.1):
        super().__init__()
        self.lam = lam

    def forward(self, logits, labels):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, labels.float())
        penalty = self.lam * (logits ** 2).mean()
        return bce + penalty

In [ ]:
# Extract embeddings
sb_splits = extract_spurbreast_features(
    fs_dir='/content/drive/MyDrive/field_strength/field_strength',
    baseline_dir='/content/drive/MyDrive/baseline_low',
    clinical_xlsx_path='/content/drive/MyDrive/Clinical_and_Other_Features.xlsx',
    rho2=0.9,
    flip_seed=42,
)

X_tr = sb_splits['train']['X']
y_tr = sb_splits['train']['y']
g_tr = sb_splits['train']['groups']
X_te = sb_splits['test']['X']
y_te = sb_splits['test']['y']
g_te = sb_splits['test']['groups']

# Bias field strength on embeddings
y_01 = ((y_tr + 1) / 2).astype(int)

# Recover z1 from groups
agree1 = g_tr // 2
z1_tr = np.where(agree1 == 1, y_01, 1 - y_01)

rng = np.random.RandomState(42)
agree_idx = np.where(y_01 == z1_tr)[0]
disagree_idx = np.where(y_01 != z1_tr)[0]

rho1 = 0.9
n_disagree_needed = int(len(agree_idx) * (1 - rho1) / rho1)

if n_disagree_needed <= len(disagree_idx):
    keep_agree = agree_idx
    keep_disagree = rng.choice(disagree_idx, size=n_disagree_needed, replace=False)
else:
    n_agree_needed = int(len(disagree_idx) * rho1 / (1 - rho1))
    keep_agree = rng.choice(agree_idx, size=n_agree_needed, replace=False)
    keep_disagree = disagree_idx

keep_idx = np.concatenate([keep_agree, keep_disagree])
rng.shuffle(keep_idx)

print(f"  Field strength biasing: {len(y_tr)} -> {len(keep_idx)} images")
print(f"  z1 correlation: {np.mean(y_01 == z1_tr):.3f} -> {np.mean(y_01[keep_idx] == z1_tr[keep_idx]):.3f}")

X_tr = X_tr[keep_idx]
y_tr = y_tr[keep_idx]
g_tr = g_tr[keep_idx]

y_tr = ((y_tr + 1) / 2).astype(np.float32)
y_te = ((y_te + 1) / 2).astype(np.float32)

sb_group_names_test = {0: 'benign', 1: 'malignant'}

In [ ]:
# ── HP sweep ──

d = X_tr.shape[1]

configs = [
    ('SD_lam005',  SpectralDecoupling(lam=0.05)),
    ('SD_lam01',   SpectralDecoupling(lam=0.1)),
    ('SD_lam05',   SpectralDecoupling(lam=0.5)),
    ('ML_lam001_g2',  MargLogLoss(lam=0.01, gamma_0=0, gamma_1=2)),
    ('ML_lam005_g2',  MargLogLoss(lam=0.05, gamma_0=0, gamma_1=2)),
    ('ML_lam01_g2',   MargLogLoss(lam=0.1,  gamma_0=0, gamma_1=2)),
    ('ML_lam05_g2',   MargLogLoss(lam=0.5,  gamma_0=0, gamma_1=2)),
    ('ML_lam1_g2',    MargLogLoss(lam=1.0,  gamma_0=0, gamma_1=2)),
    ('ML_g1_05',   MargLogLoss(lam=0.1, gamma_0=0, gamma_1=0.5)),
    ('ML_g1_1',    MargLogLoss(lam=0.1, gamma_0=0, gamma_1=1)),
    ('ML_g1_2',    MargLogLoss(lam=0.1, gamma_0=0, gamma_1=2)),
    ('σD_T1_05',  SigmaDampLoss(T_0=1, T_1=0.5)),
    ('σD_T1_1',   SigmaDampLoss(T_0=1, T_1=1)),
    ('σD_T1_2',   SigmaDampLoss(T_0=1, T_1=2)),
    ('σD_T1_3',   SigmaDampLoss(T_0=1, T_1=3)),
    ('σD_T0_05',  SigmaDampLoss(T_0=0.5, T_1=2)),
    ('σD_T0_1',   SigmaDampLoss(T_0=1,   T_1=2)),
    ('σD_T0_2',   SigmaDampLoss(T_0=2,   T_1=2)),
]

hp_results = []

for config_name, loss_factory in configs:
    print(f"\n{'='*60}")
    print(f"  SpurBreast HP: {config_name}")
    print(f"{'='*60}")

    torch.manual_seed(0)
    model = nn.Linear(d, 1, bias=False).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    Xt = torch.tensor(X_tr, dtype=torch.float32).to(device)
    yt = torch.tensor(y_tr, dtype=torch.float32).to(device)

    loss_fn = loss_factory
    for epoch in range(5000):
        optimizer.zero_grad()
        logits = model(Xt).squeeze()
        loss = loss_fn(logits, yt)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        test_logits = model(torch.tensor(X_te, dtype=torch.float32).to(device)).squeeze().cpu().numpy()
        test_preds = (test_logits > 0).astype(float)

    metrics = compute_group_metrics(test_preds, y_te, g_te, sb_group_names_test)
    print(f"  => avg_acc={metrics['avg_acc']:.3f} | wga={metrics['wga']:.3f}")

    hp_results.append({
        'config': config_name,
        'avg_acc': metrics['avg_acc'],
        'wga': metrics['wga'],
    })

df_hp_sb = pd.DataFrame(hp_results)
save_csv(df_hp_sb, 'results_spurbreast_hyperparam_linear.csv')
print(df_hp_sb.to_string(index=False, float_format='{:.3f}'.format))

---
## 12. SpurBreast — Hyperparameter Tuning (Finetuning)

In [ ]:
# ── SpurBreast Dataset classes for finetuning ──

class SpurBreastDataset(Dataset):
    """PyTorch Dataset that loads images on-the-fly with shortcut metadata."""
    def __init__(self, image_paths, labels, z1_labels, z2_labels, groups, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.z1_labels = z1_labels
        self.z2_labels = z2_labels
        self.groups = groups
        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        if self.z2_labels[idx] == 1:
            img = img.transpose(Image.FLIP_TOP_BOTTOM)
        img = self.transform(img)
        return img, self.labels[idx], self.groups[idx]


class SpurBreastTestDataset(Dataset):
    """Test dataset — no shortcuts applied, just clean images."""
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        img = self.transform(img)
        return img, self.labels[idx], self.labels[idx]


def prepare_spurbreast_datasets(fs_dir, baseline_dir, clinical_xlsx_path,
                                 rho2=0.9, flip_seed=42):
    """Prepare SpurBreast datasets for finetuning (images loaded on-the-fly)."""
    fs_map = load_duke_field_strength(clinical_xlsx_path)

    def get_fs_label(filename):
        try:
            pat_num = int(os.path.basename(filename).split('-')[1])
            return fs_map.get(pat_num, 0)
        except:
            return 0

    # Collect training paths + metadata
    train_dir = os.path.join(fs_dir, 'training')
    pos_paths = get_file_paths(os.path.join(train_dir, '1'), 1)
    neg_paths = get_file_paths(os.path.join(train_dir, '0'), 0)
    all_paths = pos_paths + neg_paths

    rng = np.random.RandomState(flip_seed)
    image_paths, y_list, z1_list, z2_list = [], [], [], []

    for path, label in all_paths:
        fname = os.path.basename(path)
        z1_val = get_fs_label(fname)
        z2_val = label if rng.random() < rho2 else 1 - label

        image_paths.append(path)
        y_list.append(label)
        z1_list.append(z1_val)
        z2_list.append(z2_val)

    y_arr = np.array(y_list, dtype=np.int64)
    z1_arr = np.array(z1_list, dtype=np.int64)
    z2_arr = np.array(z2_list, dtype=np.int64)

    # Bias field strength correlation
    image_paths, y_arr, z1_arr, z2_arr = bias_field_strength(
        image_paths, y_arr, z1_arr, z2_arr, rho1=0.9, seed=flip_seed
    )

    # Recompute groups after biasing
    agree1 = (y_arr == z1_arr).astype(int)
    agree2 = (y_arr == z2_arr).astype(int)
    groups_arr = 2 * agree1 + agree2

    print(f"Training: {len(y_arr)} images")
    print(f"  z1 (field strength) correlation: {np.mean(agree1):.3f}")
    print(f"  z2 (vertical flip) correlation:  {np.mean(agree2):.3f}")
    print(f"  Groups: {dict(zip(*np.unique(groups_arr, return_counts=True)))}")

    # 80/20 train/val split
    np.random.seed(42)
    indices = np.random.permutation(len(y_arr))
    split = int(len(y_arr) * 0.8)
    tr_idx, val_idx = indices[:split], indices[split:]

    train_dataset = SpurBreastDataset(
        [image_paths[i] for i in tr_idx],
        y_arr[tr_idx], z1_arr[tr_idx], z2_arr[tr_idx], groups_arr[tr_idx]
    )

    # Test set (baseline, unbiased)
    test_dir = os.path.join(baseline_dir, 'test')
    test_pos = get_file_paths(os.path.join(test_dir, '1'), 1)
    test_neg = get_file_paths(os.path.join(test_dir, '0'), 0)
    test_all = test_pos + test_neg
    test_paths = [p for p, _ in test_all]
    test_labels = np.array([l for _, l in test_all], dtype=np.int64)
    test_dataset = SpurBreastTestDataset(test_paths, test_labels)

    # 20% of test as unbiased validation for early stopping
    np.random.seed(99)
    test_indices = np.random.permutation(len(test_labels))
    val_split = int(len(test_labels) * 0.2)

    unbiased_val_dataset = SpurBreastTestDataset(
        [test_paths[i] for i in test_indices[:val_split]],
        test_labels[test_indices[:val_split]]
    )
    final_test_dataset = SpurBreastTestDataset(
        [test_paths[i] for i in test_indices[val_split:]],
        test_labels[test_indices[val_split:]]
    )
    print(f"Test: {len(test_labels)} images")

    return train_dataset, unbiased_val_dataset, final_test_dataset

In [ ]:
# ── Finetuning helpers for SpurBreast ──

def create_resnet_model(device='cuda'):
    """Fresh pretrained ResNet-50 with a single output head."""
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    model.fc = nn.Linear(2048, 1)
    return model.to(device)


def evaluate_model(model, loader, device='cuda'):
    """Evaluate model, return avg acc, WGA, and per-group accs."""
    model.eval()
    all_preds, all_labels, all_groups = [], [], []

    with torch.no_grad():
        for x, y, g in loader:
            x = x.to(device)
            logits = model(x).squeeze(-1)
            preds = (logits > 0).long().cpu()
            all_preds.append(preds)
            all_labels.append(y)
            all_groups.append(g)

    preds = torch.cat(all_preds)
    labels = torch.cat(all_labels).long()
    groups = torch.cat(all_groups)

    correct = (preds == labels).float()
    avg_acc = correct.mean().item()

    group_accs = {}
    for g in groups.unique():
        mask = groups == g
        if mask.sum() > 0:
            group_accs[g.item()] = correct[mask].mean().item()

    wga = min(group_accs.values()) if group_accs else 0.0
    return avg_acc, wga, group_accs


def train_finetune_spurbreast(model, train_loader, val_loader, loss_fn,
                               lr=1e-5, weight_decay=0.1, epochs=50,
                               patience=10, device='cuda'):
    """Full finetuning with early stopping on validation WGA."""
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val_wga = -1
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        n_samples = 0
        for x, y, g in train_loader:
            x, y = x.to(device), y.to(device).float()
            optimizer.zero_grad()
            logits = model(x).squeeze(-1)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)
            n_samples += x.size(0)

        val_avg, val_wga, _ = evaluate_model(model, val_loader, device)

        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}/{epochs} | "
                  f"train_loss={total_loss/n_samples:.4f} | "
                  f"val_avg={val_avg:.3f} | val_wga={val_wga:.3f}")

        if val_wga > best_val_wga:
            best_val_wga = val_wga
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_val_wga

In [ ]:
# ── Run SpurBreast HP sweep (finetuning) ──

def run_hyperparam_sweep(train_dataset, val_dataset, test_dataset, device='cuda'):
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

    d = 2048

    configs = [
        ('SD_lam005',  SpectralDecoupling(lam=0.05)),
        ('SD_lam01',   SpectralDecoupling(lam=0.1)),
        ('SD_lam05',   SpectralDecoupling(lam=0.5)),
        ('ML_lam001_g2',  MargLogLoss(lam=0.01, gamma_0=0, gamma_1=2)),
        ('ML_lam005_g2',  MargLogLoss(lam=0.05, gamma_0=0, gamma_1=2)),
        ('ML_lam01_g2',   MargLogLoss(lam=0.1,  gamma_0=0, gamma_1=2)),
        ('ML_lam05_g2',   MargLogLoss(lam=0.5,  gamma_0=0, gamma_1=2)),
        ('ML_lam1_g2',    MargLogLoss(lam=1.0,  gamma_0=0, gamma_1=2)),
        ('ML_g1_05',   MargLogLoss(lam=0.1, gamma_0=0, gamma_1=0.5)),
        ('ML_g1_1',    MargLogLoss(lam=0.1, gamma_0=0, gamma_1=1)),
        ('ML_g1_2',    MargLogLoss(lam=0.1, gamma_0=0, gamma_1=2)),
        ('σD_T1_05',  SigmaDampLoss(T_0=1, T_1=0.5)),
        ('σD_T1_1',   SigmaDampLoss(T_0=1, T_1=1)),
        ('σD_T1_2',   SigmaDampLoss(T_0=1, T_1=2)),
        ('σD_T1_3',   SigmaDampLoss(T_0=1, T_1=3)),
        ('σD_T0_05',  SigmaDampLoss(T_0=0.5, T_1=2)),
        ('σD_T0_1',   SigmaDampLoss(T_0=1,   T_1=2)),
        ('σD_T0_2',   SigmaDampLoss(T_0=2,   T_1=2)),
    ]

    all_results = []

    for config_name, loss_fn in configs:
        print(f"\n{'='*60}")
        print(f"  Hyperparam sweep: {config_name}")
        print(f"{'='*60}")

        torch.manual_seed(0)
        model = create_resnet_model(device)

        train_loader = DataLoader(train_dataset, batch_size=64,
                                  shuffle=True, num_workers=4, pin_memory=True)

        model, val_wga = train_finetune_spurbreast(
            model, train_loader, val_loader, loss_fn,
            lr=1e-5, weight_decay=0.1, epochs=50, patience=20, device=device
        )

        test_avg, test_wga, test_groups = evaluate_model(model, test_loader, device)
        print(f"  => val_wga={val_wga:.3f} | test_wga={test_wga:.3f} | test_avg={test_avg:.3f}")

        all_results.append({
            'config': config_name,
            'val_wga': val_wga,
            'test_wga': test_wga,
            'test_avg': test_avg,
        })

    return pd.DataFrame(all_results)

# Prepare datasets and run
train_dataset, val_dataset, test_dataset = prepare_spurbreast_datasets(
    fs_dir='/content/local_data/field_strength',
    baseline_dir='/content/local_data/baseline_low',
    clinical_xlsx_path='/content/local_data/Clinical_and_Other_Features.xlsx',
    rho2=0.9,
    flip_seed=42,
)
df_hp = run_hyperparam_sweep(train_dataset, val_dataset, test_dataset, device='cuda')
DRIVE_ROOT = '/content/drive/MyDrive/shortcut_experiments'
os.makedirs(f'{DRIVE_ROOT}/csv', exist_ok=True)
df_hp.to_csv(f'{DRIVE_ROOT}/csv/results_spurbreast_hyperparam.csv', index=False)
print(df_hp.to_string(index=False, float_format='{:.3f}'.format))

---
## 13. SpurBreast — Testing (Linear Probe)

Using best hyperparameters from Section 11.

In [ ]:
# ── Linear probe testing functions for SpurBreast ──

# Redefine run_linear_probe_fixed for SpurBreast (same structure, different data)
# Uses the version defined in Section 5 but with SpurBreast-specific loss configs

def get_loss_methods():
    return {
        'ERM': LogLoss(),
        'SD': SpectralDecoupling(lam=0.05),
        'Marg-Log': MargLogLoss(lam=0.5, gamma_0=0, gamma_1=2),
        'σ-damp': SigmaDampLoss(T_0=0.5, T_1=2),
    }


def run_fixed_subsample(embeddings, device='cuda'):
    """Run linear probe with subsampling strategies."""
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(embeddings['train']['X']).astype(np.float32)
    X_val = scaler.transform(embeddings['val']['X']).astype(np.float32)
    X_te = scaler.transform(embeddings['test']['X']).astype(np.float32)

    y_tr, g_tr = embeddings['train']['y'], embeddings['train']['groups']

    conditions = ['Full', 'GB-Down', 'GB-Over', 'Random (r=0.25)']
    methods = get_loss_methods()
    all_results = []

    for cond in conditions:
        if cond == 'Full': X_c, y_c, g_c = X_tr, y_tr, g_tr
        elif cond == 'GB-Down': X_c, y_c, g_c = gb_downsample(X_tr, y_tr, g_tr)
        elif cond == 'GB-Over': X_c, y_c, g_c = gb_oversample(X_tr, y_tr, g_tr)
        elif cond == 'Random (r=0.25)':
            idx = np.random.choice(len(y_tr), size=len(y_tr)//4, replace=False)
            X_c, y_c, g_c = X_tr[idx], y_tr[idx], g_tr[idx]

        print(f"\n{'='*60}\n  {cond}: n={len(y_c)}, d/n={X_c.shape[1]/len(y_c):.3f}\n{'='*60}")

        for name, loss_fn in methods.items():
            seed_wgas = []
            for seed in range(3):
                np.random.seed(seed); torch.manual_seed(seed)
                res = run_linear_probe_fixed(X_c, y_c, g_c, X_val, embeddings['val']['y'], embeddings['val']['groups'],
                                             X_te, embeddings['test']['y'], embeddings['test']['groups'],
                                             loss_fn, device=device)
                seed_wgas.append(res['test_wga'])
                all_results.append({'condition': cond, 'method': name, 'seed': seed, 'test_wga': res['test_wga'], 'd_over_n': X_c.shape[1]/len(y_c)})
            print(f"  {name:>10s} | WGA = {np.mean(seed_wgas):.3f} ± {np.std(seed_wgas):.3f}")
    return pd.DataFrame(all_results)

In [ ]:
# Run SpurBreast linear probe testing
# Re-extract embeddings for clean run
sb_embeddings = extract_spurbreast_features(
    fs_dir='/content/local_data/field_strength',
    baseline_dir='/content/local_data/baseline_low',
    clinical_xlsx_path='/content/local_data/Clinical_and_Other_Features.xlsx',
    rho2=0.9,
    device=device
)

df_sb_lp = run_fixed_subsample(sb_embeddings, device)
df_sb_lp.to_csv('spurbreast_subsample.csv', index=False)

---
## 14. SpurBreast — Testing (Finetuning)

Using best hyperparameters from Section 12.

In [ ]:
# ── Subsampling for finetuning datasets ──

def get_subset_by_indices(dataset, indices):
    return Subset(dataset, indices)


def subsample_dataset(dataset, strategy='full', seed=0):
    """Apply subsampling strategy to a SpurBreastDataset."""
    groups = dataset.groups
    n = len(groups)
    rng = np.random.RandomState(seed)

    if strategy == 'full':
        return dataset

    elif strategy == 'gb_down':
        unique_groups = np.unique(groups)
        min_size = min(np.sum(groups == g) for g in unique_groups)
        indices = []
        for g in unique_groups:
            g_idx = np.where(groups == g)[0]
            chosen = rng.choice(g_idx, size=min_size, replace=False)
            indices.extend(chosen)
        rng.shuffle(indices)
        return get_subset_by_indices(dataset, indices)

    elif strategy.startswith('gb_down_'):
        frac = int(strategy.split('_')[2]) / 100
        n_keep = max(int(n * frac), 2)
        pre_idx = rng.choice(n, size=n_keep, replace=False)
        pre_groups = groups[pre_idx]
        unique_groups = np.unique(pre_groups)
        min_size = min(np.sum(pre_groups == g) for g in unique_groups)
        if min_size == 0:
            return get_subset_by_indices(dataset, pre_idx)
        indices = []
        for g in unique_groups:
            g_idx = np.where(pre_groups == g)[0]
            chosen = rng.choice(g_idx, size=min_size, replace=False)
            indices.extend(chosen)
        rng.shuffle(indices)
        final_idx = pre_idx[indices]
        return get_subset_by_indices(dataset, final_idx)

    elif strategy == 'gb_over':
        unique_groups = np.unique(groups)
        max_size = max(np.sum(groups == g) for g in unique_groups)
        indices = []
        for g in unique_groups:
            g_idx = np.where(groups == g)[0]
            chosen = rng.choice(g_idx, size=max_size, replace=True)
            indices.extend(chosen)
        rng.shuffle(indices)
        return get_subset_by_indices(dataset, indices)

    elif strategy.startswith('random_'):
        ratio = int(strategy.split('_')[1]) / 100
        n_keep = max(int(n * ratio), 2)
        indices = rng.choice(n, size=n_keep, replace=False)
        return get_subset_by_indices(dataset, indices)

    elif strategy == 'importance_weighted':
        unique_groups = np.unique(groups)
        n_per_group = n // len(unique_groups)
        indices = []
        for g in unique_groups:
            g_idx = np.where(groups == g)[0]
            chosen = rng.choice(g_idx, size=n_per_group, replace=True)
            indices.extend(chosen)
        rng.shuffle(indices)
        return get_subset_by_indices(dataset, indices)

    else:
        raise ValueError(f"Unknown strategy: {strategy}")


def get_shortcut_ratio(dataset):
    """Compute fraction of samples where each shortcut agrees with label."""
    if isinstance(dataset, Subset):
        parent = dataset.dataset
        indices = dataset.indices
        groups = parent.groups[indices]
    else:
        groups = dataset.groups

    z1_agree = np.mean(np.isin(groups, [2, 3]))
    z2_agree = np.mean(np.isin(groups, [1, 3]))
    return z1_agree, z2_agree

In [ ]:
def run_spurbreast_finetuning(train_dataset, val_dataset, test_dataset,
                               device='cuda'):
    """Full finetuning experiment across conditions and methods."""
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

    d = 2048

    methods = {
        'ERM':      LogLoss(),
        'SD':       SpectralDecoupling(lam=0.05),
        'Marg-Log': MargLogLoss(lam=0.5, gamma_0=0, gamma_1=2),
        'σ-damp':   SigmaDampLoss(T_0=1, T_1=0.5),
    }

    conditions = [
        'random_005', 'random_010', 'random_025', 'random_050', 'full',
        'gb_down', 'gb_down_025', 'gb_down_050', 'gb_down_075', 'gb_over',
        'importance_weighted',
    ]

    all_results = []

    for cond in conditions:
        for method_name, loss_fn in methods.items():
            for seed in range(2):
                print(f"\n{'='*60}")
                print(f"  {cond} | {method_name} | seed={seed}")
                print(f"{'='*60}")

                sub_dataset = subsample_dataset(train_dataset, strategy=cond, seed=seed)
                n_train = len(sub_dataset)
                d_over_n = d / n_train
                z1_rho, z2_rho = get_shortcut_ratio(sub_dataset)

                print(f"  n_train={n_train}, d/n={d_over_n:.3f}")
                print(f"  z1 shortcut-label ratio={z1_rho:.3f}, z2={z2_rho:.3f}")

                train_loader = DataLoader(sub_dataset, batch_size=64,
                                          shuffle=True, num_workers=4)

                torch.manual_seed(seed)
                model = create_resnet_model(device)

                model, val_wga = train_finetune_spurbreast(
                    model, train_loader, val_loader, loss_fn,
                    lr=1e-5,
                    weight_decay=0.1,
                    epochs=50,
                    patience=20,
                    device=device
                )

                test_avg, test_wga, test_groups = evaluate_model(
                    model, test_loader, device
                )

                print(f"  => test_avg={test_avg:.3f} | test_wga={test_wga:.3f}")

                all_results.append({
                    'condition': cond,
                    'method': method_name,
                    'seed': seed,
                    'n_train': n_train,
                    'd': d,
                    'd_over_n': d_over_n,
                    'z1_shortcut_ratio': z1_rho,
                    'z2_shortcut_ratio': z2_rho,
                    'val_wga': val_wga,
                    'test_wga': test_wga,
                    'test_avg': test_avg,
                })

    return pd.DataFrame(all_results)

In [ ]:
# ── Run SpurBreast finetuning test ──

import shutil

# Copy data to local SSD (faster I/O)
if not os.path.exists('/content/local_data/field_strength'):
    print("Copying data to local disk...")
    shutil.copytree('/content/drive/MyDrive/field_strength/field_strength', '/content/local_data/field_strength')
    shutil.copytree('/content/drive/MyDrive/baseline_low', '/content/local_data/baseline_low')
    shutil.copy('/content/drive/MyDrive/Clinical_and_Other_Features.xlsx', '/content/local_data/Clinical_and_Other_Features.xlsx')
    print("Done copying.")

train_dataset, val_dataset, test_dataset = prepare_spurbreast_datasets(
    fs_dir='/content/local_data/field_strength',
    baseline_dir='/content/local_data/baseline_low',
    clinical_xlsx_path='/content/local_data/Clinical_and_Other_Features.xlsx',
    rho2=0.9,
    flip_seed=42,
)

df_spurbreast_ft = run_spurbreast_finetuning(
    train_dataset, val_dataset, test_dataset,
    device='cuda'
)

# Save results
DRIVE_ROOT = '/content/drive/MyDrive/shortcut_experiments'
df_spurbreast_ft.to_csv(f'{DRIVE_ROOT}/csv/results_spurbreast_finetuning.csv', index=False)

# Print summary
agg = df_spurbreast_ft.groupby(['condition', 'method']).agg(
    test_wga=('test_wga', 'mean'),
    test_wga_std=('test_wga', 'std'),
).reset_index()
print(agg.to_string(index=False, float_format='{:.3f}'.format))